In [2]:
import numpy as np
import pandas as pd
import awkward as ak
import uproot
from functools import reduce

In [3]:
import matplotlib.pyplot as plt
from IPython.display import Image, display
from matplotlib.colors import LogNorm
# Aesthetics:
fs = 14    # fontsize

In [4]:
# Check out the structure
path = "/home/pira/Documenti/PoD/LCP/LCP_B/ALICE/AO2DtreeMC.root"
file = uproot.open(path)
file.classnames()

{'DF_2303121152302944;1': 'TDirectory',
 'DF_2303121152302944/O2mccollision;1': 'TTree',
 'DF_2303121152302944/O2collision_001;1': 'TTree',
 'DF_2303121152302944/O2filtertrack;1': 'TTree',
 'DF_2303121152302944/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302944/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302944/O2genparticles;1': 'TTree',
 'DF_2303121152302976;1': 'TDirectory',
 'DF_2303121152302976/O2mccollision;1': 'TTree',
 'DF_2303121152302976/O2collision_001;1': 'TTree',
 'DF_2303121152302976/O2filtertrack;1': 'TTree',
 'DF_2303121152302976/O2filtertrackextr;1': 'TTree',
 'DF_2303121152302976/O2filtertrackmc;1': 'TTree',
 'DF_2303121152302976/O2genparticles;1': 'TTree',
 'DF_2303121152303008;1': 'TDirectory',
 'DF_2303121152303008/O2mccollision;1': 'TTree',
 'DF_2303121152303008/O2collision_001;1': 'TTree',
 'DF_2303121152303008/O2filtertrack;1': 'TTree',
 'DF_2303121152303008/O2filtertrackextr;1': 'TTree',
 'DF_2303121152303008/O2filtertrackmc;1': 'TTree',
 'DF_2303121152303008

In [5]:
test = file["DF_2303121152302944/O2filtertrack"].arrays(library="pd", entry_stop=15)
test.columns

Index(['fIndexCollisions', 'fIsInsideBeamPipe', 'fTrackType', 'fX', 'fAlpha',
       'fY', 'fZ', 'fSnp', 'fTgl', 'fSigned1Pt'],
      dtype='object')

In [6]:
test = file["DF_2303121152302944/O2filtertrackmc"].arrays(library="pd", entry_stop=15)
test.columns

Index(['fPdgCode', 'fIsPhysicalPrimary', 'fMainHfMotherPdgCode',
       'fMainBeautyAncestorPdgCode', 'fMainMotherOrigIndex',
       'fMainMotherNfinalStateDaught', 'fMainMotherPt', 'fMainMotherY',
       'fMainBeautyAncestorPt', 'fMainBeautyAncestorY'],
      dtype='object')

In [7]:
# Function to filter out the "good" couples of tracks

def check_group(group):
    part_pdg = sorted(group['fPdgCode'].tolist())
    return len(group) == 2 and (
        part_pdg == [-321, 211] or  # D⁰ → K⁻ π⁺
        part_pdg == [-211, 321]     # D̄⁰ → π⁻ K⁺
    )

def is_D0(group):
    return sorted(group['fPdgCode'].tolist()) == [-321, 211]

def is_D0bar(group):
    return sorted(group['fPdgCode'].tolist()) == [-211, 321]


In [8]:
#parameters for the cuts
acceptance = 0.8

# Pt intervals definition
Pt_intervals = [0, 1, 2, 3, 5, 8, 41]
Pt_labels = ["0-1", "1-2", "2-3", "3-5", "5-8", "8-41"]

## Reconstructed particles

In [9]:
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]

#good_gen_part = pd.DataFrame()
#tot_gen_part = pd.DataFrame()
# Variable to count the reconstructed particles
n_D_0 = 0
n_D_0_bar = 0
tot_decays = 0
# Empty list to store all the dataframes of the tracks after the selections
filtered_tracks = []


for directory in file_ID:

    # Import the data and merge the dataframes
    tracks_df = file[directory + "/O2filtertrack"].arrays(["fIndexCollisions", 
                                                           "fX", 
                                                           "fAlpha",
                                                           "fY", 
                                                           "fZ"], library="pd")
    
    tracks_mc_df = file[directory + "/O2filtertrackmc"].arrays(["fPdgCode",
                                                                "fMainMotherOrigIndex",
                                                                "fMainHfMotherPdgCode",
                                                                "fMainMotherNfinalStateDaught",
                                                                "fMainBeautyAncestorPdgCode"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_mc_df, 
                         how='inner', left_index=True, right_index=True)

    # Merge the "extra" table to obtain the "eta" variable
    #tracks_extra = file[directory + "/O2filtertrackextr"].arrays(["fEta"],library="pd")
    #tracks_df = pd.merge(left=tracks_df, 
                         #right=tracks_extra, 
                         #how='inner', left_index=True, right_index=True)

    # Merge the "extra" table to obtain the "eta" and "Pt" variables
    tracks_extra = file[directory + "/O2filtertrackextr"].arrays(["fEta", "fPt"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_extra, 
                         how='inner', left_index=True, right_index=True)

    # Merge the collision table to perform a cut on fPosZ
    collision_fPosZ = file[directory + "/O2collision_001"].arrays( ["fPosZ"], library="pd")
    tracks_df = pd.merge(left = tracks_df, right = collision_fPosZ, how = 'inner', left_on = 'fIndexCollisions', right_index = True)

    # fPosZ cut
    tracks_df = tracks_df[tracks_df["fPosZ"].abs() < 10]

    # Select the right particles (fPdgCode)
    #tracks_df = tracks_df[tracks_df["fPdgCode"].isin([-321,211])]
    
    # Select all the rows with fMainHfMotherPdgCode = 421 AND fMainMotherNfinalStateDaught = 2
    tracks_df = tracks_df[(tracks_df["fMainHfMotherPdgCode"] == 421) 
                        & (tracks_df['fMainMotherNfinalStateDaught'] >= 2)]

    # Pseudorapidity cut
    tracks_df = tracks_df[tracks_df["fEta"].abs() < acceptance]

    # Check on the fMainBeautyAncestor = 0
    tracks_df = tracks_df[tracks_df["fMainBeautyAncestorPdgCode"] == 0]

    #defined pT interval in which you can/want to “make the measurement”
    #tracks_df["interval"] = pd.cut(tracks_df['fPt'], bins=Pt_intervals, labels=Pt_labels, right=True)
    #r = tracks_df.groupby("interval", observed=True).size()
    #filtered_tracks_Pt.append(r)
    
    # fMainMotherOrigIndex is the same for the grouped tracks
    tracks_df = tracks_df.groupby('fMainMotherOrigIndex')

    D_0_decays = tracks_df.filter(is_D0)
    D_0_bar_decays = tracks_df.filter(is_D0bar)
    
    n_D_0 = len(D_0_decays) // 2
    n_D_0_bar = len(D_0_bar_decays) // 2
    
    #print(f"D⁰: {n_D_0} | D̄⁰: {n_D_0_bar}")
   
    # Applica il filtro sui gruppi
    #D_0_decays = tracks_df.filter(check_group)


    # Contiamo le D_0 e sommiamole per avere un totale
    tot_decays += (n_D_0 + n_D_0_bar)

    #print("In this file there are", len(D_0_decays)/2, "D_0 decays")

    filtered_tracks.append(D_0_decays)
    
    

In [10]:
tot_decays

1271

In [11]:
filtered_tracks[1].head(10)

,fIndexCollisions,fX,fAlpha,fY,fZ,fPdgCode,fMainMotherOrigIndex,fMainHfMotherPdgCode,fMainMotherNfinalStateDaught,fMainBeautyAncestorPdgCode,fEta,fPt,fPosZ
1743,222,-0.036868,1.240051,0.027763,-6.793315,211,398484,421,2,0,0.764998,1.208464,-6.798203
1745,222,0.019147,-1.397227,-0.026286,-6.793478,-321,398484,421,2,0,-0.727157,0.318547,-6.798203
2625,345,0.002446,-1.052740,-0.036397,5.865180,211,604014,421,2,0,0.480556,7.202230,5.866066
2626,345,-0.006498,-0.818237,-0.044119,5.879215,-321,604014,421,2,0,0.117874,0.919719,5.866066


In [12]:
#Copy
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]

#good_gen_part = pd.DataFrame()
#tot_gen_part = pd.DataFrame()
# Variable to count the reconstructed particles
n_D_0 = 0
n_D_0_bar = 0
tot_decays = 0

# Empty list to store all the dataframes of the tracks after the selections
reconstructed_part = []


for directory in file_ID:

    # Import the data and merge the dataframes
    tracks_df = file[directory + "/O2filtertrack"].arrays(["fIndexCollisions", 
                                                           "fX", 
                                                           "fAlpha",
                                                           "fY", 
                                                           "fZ"], library="pd")
    
    tracks_mc_df = file[directory + "/O2filtertrackmc"].arrays(["fPdgCode", 
                                                                "fMainMotherOrigIndex", 
                                                                "fMainHfMotherPdgCode", 
                                                                "fMainMotherNfinalStateDaught", 
                                                                "fMainBeautyAncestorPdgCode"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_mc_df, 
                         how='inner', left_index=True, right_index=True)

    # Merge the "extra" table to obtain the "eta" variable
    #tracks_extra = file[directory + "/O2filtertrackextr"].arrays(["fEta"],library="pd")
    #tracks_df = pd.merge(left=tracks_df, 
                         #right=tracks_extra, 
                         #how='inner', left_index=True, right_index=True)

    # Merge the "extra" table to obtain the "eta" and "Pt" variables
    tracks_extra = file[directory + "/O2filtertrackextr"].arrays(["fEta", "fPt"],library="pd")
    tracks_df = pd.merge(left=tracks_df, 
                         right=tracks_extra, 
                         how='inner', left_index=True, right_index=True)

    # Merge the collision table to perform a cut on fPosZ
    collision_fPosZ = file[directory + "/O2collision_001"].arrays( ["fPosZ"], library="pd")
    tracks_df = pd.merge(left = tracks_df, right = collision_fPosZ, how = 'inner', left_on = 'fIndexCollisions', right_index = True)

    # fPosZ cut
    tracks_df = tracks_df[tracks_df["fPosZ"].abs() < 10]

    # Select the right particles (fPdgCode)
    tracks_df = tracks_df[tracks_df["fPdgCode"].isin([-321,211,321,-211])]
    
    # Select all the rows with fMainHfMotherPdgCode = 421 OR -421 AND fMainMotherNfinalStateDaught = 2
    tracks_df = tracks_df[(tracks_df["fMainHfMotherPdgCode"].isin([421, -421]))
                       & (tracks_df['fMainMotherNfinalStateDaught'] >= 2)]

    # Pseudorapidity cut
    tracks_df = tracks_df[tracks_df["fEta"].abs() < acceptance]

    # Check on the fMainBeautyAncestor = 0
    tracks_df = tracks_df[tracks_df["fMainBeautyAncestorPdgCode"] == 0]

    #defined pT interval in which you can/want to “make the measurement”
    #tracks_df["interval"] = pd.cut(tracks_df['fPt'], bins=Pt_intervals, labels=Pt_labels, right=True)
    #r = tracks_df.groupby("interval", observed=True).size()
    #filtered_tracks_Pt.append(r)
    
    # fMainMotherOrigIndex is the same for the grouped tracks
    #tracks_df = tracks_df.groupby('fMainMotherOrigIndex')

    #D_0_decays = tracks_df.filter(is_D0)
    #D_0_bar_decays = tracks_df.filter(is_D0bar)
    
    #n_D_0 = len(D_0_decays) // 2
    #n_D_0_bar = len(D_0_bar_decays) // 2
    
    #print(f"D⁰: {n_D_0} | D̄⁰: {n_D_0_bar}")
   
    # Applica il filtro sui gruppi
    #D_0_decays = tracks_df.filter(check_group)


    # Contiamo le D_0 e sommiamole per avere un totale
    #tot_decays += (n_D_0 + n_D_0_bar)

    #print("In this file there are", len(D_0_decays)/2, "D_0 decays")

    reconstructed_part.append(tracks_df)
    

In [13]:
reconstructed_part[1]

,fIndexCollisions,fX,fAlpha,fY,fZ,fPdgCode,fMainMotherOrigIndex,fMainHfMotherPdgCode,fMainMotherNfinalStateDaught,fMainBeautyAncestorPdgCode,fEta,fPt,fPosZ
67,14,0.046524,3.090803,0.020455,-5.476096,-211,20456,-421,2,0,0.619300,0.946830,-5.473442
534,74,-0.029238,0.149362,-0.007702,1.574896,-211,129871,-421,2,0,0.051778,2.775677,1.571714
578,79,-0.036794,-0.309142,-0.027292,-2.973794,-321,138052,421,2,0,-0.054345,0.442877,-2.973785
1048,136,0.002623,-0.963504,-0.033178,-3.418107,321,237489,-421,2,0,-0.781936,2.681206,-3.411430
1083,141,0.041651,-1.763925,-0.030461,4.827859,211,246671,421,2,0,-0.108136,3.722790,-2.135948
1292,163,0.028047,-1.542282,-0.080748,-4.967772,321,297070,-421,2,0,0.320548,0.627449,0.601987
1386,175,-0.053462,0.362196,-0.009065,7.131252,321,311078,-421,2,0,0.752127,0.937482,7.110733
1387,175,-0.009065,2.176565,0.045943,7.122959,-211,311078,-421,2,0,0.778457,1.262795,7.110733
1682,212,-0.013832,-0.526057,-0.042273,-0.495630,-321,381656,421,2,0,-0.660484,1.768880,-0.490324
1720,218,0.035067,-1.959452,-0.023728,-6.259479,-211,391986,-421,2,0,-0.362436,1.048406,-6.257690


In [14]:
good_tracks = pd.concat(reconstructed_part)

In [15]:
good_tracks.head(10)

,fIndexCollisions,fX,fAlpha,fY,fZ,fPdgCode,fMainMotherOrigIndex,fMainHfMotherPdgCode,fMainMotherNfinalStateDaught,fMainBeautyAncestorPdgCode,fEta,fPt,fPosZ
133,17,0.039578,-2.839309,0.014309,-4.003684,-211,32119,-421,2,0,-0.658791,1.364974,-4.008034
547,72,-0.039726,0.444647,-0.006729,-0.698118,-211,132931,-421,2,0,-0.275567,0.638932,-0.696275
686,87,-0.004605,2.093731,0.031284,2.639622,-211,166221,-421,2,0,-0.633307,1.711817,2.630234
694,87,0.000643,2.235773,0.051432,2.621585,211,165804,421,2,0,-0.330267,1.808290,2.630234
995,124,-0.038186,0.634623,-0.000120,4.492827,211,236063,421,2,0,0.022556,6.100133,4.493095
1015,128,0.010831,2.397087,-0.003807,4.338156,211,244089,421,2,0,0.271339,0.414826,3.843246
1025,130,-0.028373,1.152111,0.027520,4.311892,-321,244089,421,2,0,0.798083,4.397004,4.311096
1310,165,0.014469,-1.406388,-0.040354,-0.394191,321,309296,-421,2,0,-0.653051,1.834698,-0.393646
1388,179,-0.016135,-0.517653,-0.026960,-3.812342,-321,330260,421,2,0,0.502536,3.741438,-3.807930
1551,200,0.007831,2.344942,0.045844,3.585036,-211,368812,-421,2,0,-0.639779,1.309054,3.586140


In [17]:
t = good_tracks.groupby("fIndexCollisions").apply("fMainMotherOrigIndex"]).size()
t

SyntaxError: closing parenthesis ']' does not match opening parenthesis '(' (3962020453.py, line 1)

In [18]:
good_tracks[good_tracks["fIndexCollisions"]==0]

,fIndexCollisions,fX,fAlpha,fY,fZ,fPdgCode,fMainMotherOrigIndex,fMainHfMotherPdgCode,fMainMotherNfinalStateDaught,fMainBeautyAncestorPdgCode,fEta,fPt,fPosZ
4,0,-0.025249,-0.393038,-0.042837,8.351005,-211,2147,-421,2,0,0.062846,0.576501,8.345566
6,0,-0.001248,2.212518,0.053894,8.349487,321,2147,-421,2,0,0.250847,1.441978,8.345566
4,0,0.023034,-1.396754,-0.048360,7.527332,-321,3984,421,2,0,-0.634710,0.740101,7.532196
442,0,0.026766,3.070175,-0.274812,4.410246,-211,14597,-421,2,0,-0.703625,0.548620,3.369495
10,0,-0.042165,0.715144,0.000410,-1.288341,211,1471,421,2,0,-0.794010,0.693413,-1.270205
13,0,0.032369,3.026426,0.024559,1.421094,-211,1295610,-421,2,0,-0.334105,0.328519,1.427170
477,0,-0.052540,0.762051,0.119159,-5.351027,211,1390267,421,2,0,-0.428675,3.195672,-5.457848
559,0,0.040463,-3.132709,0.032760,0.860536,211,5910,421,2,0,0.331264,1.928061,0.987932
6,0,0.031458,-2.004936,-0.020787,3.872896,211,1297645,421,2,0,0.470555,2.332052,3.874798
534,0,-0.043726,0.671519,0.145468,5.059761,-321,1401699,421,2,0,0.644521,0.749059,4.838257


In [19]:
# Per ogni collisione teniamo solo le tracce che hanno la madre che viene dallo stesso punto
selected_tracks = good_tracks.groupby('fIndexCollisions', group_keys=False).apply(
    lambda g: g[g['fMainMotherOrigIndex'].isin(
        g['fMainMotherOrigIndex'].value_counts()[g['fMainMotherOrigIndex'].value_counts() >= 2].index
    )]
)

selected_tracks

/tmp/ipykernel_12997/2672277737.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  selected_tracks = good_tracks.groupby('fIndexCollisions', group_keys=False).apply(


,fIndexCollisions,fX,fAlpha,fY,fZ,fPdgCode,fMainMotherOrigIndex,fMainHfMotherPdgCode,fMainMotherNfinalStateDaught,fMainBeautyAncestorPdgCode,fEta,fPt,fPosZ
4,0,-0.025249,-0.393038,-0.042837,8.351005,-211,2147,-421,2,0,0.062846,0.576501,8.345566
6,0,-0.001248,2.212518,0.053894,8.349487,321,2147,-421,2,0,0.250847,1.441978,8.345566
2,0,-0.036042,1.312898,0.044301,-5.631733,321,5132,-421,2,0,0.308542,1.305689,-5.619606
11,0,-0.012715,1.955349,0.036739,-5.615656,-211,5132,-421,2,0,-0.190017,3.143549,-5.619606
38,1,-0.046642,0.061536,-0.025308,4.193429,211,4656,421,2,0,-0.454421,3.220495,4.181335
...,...,...,...,...,...,...,...,...,...,...,...,...,...
11126,1477,-0.000692,-0.939444,-0.048250,3.023618,321,2608448,-421,2,0,0.103468,1.751295,3.028175
11171,1484,-0.015587,-0.219195,-0.024905,7.570144,-211,2623785,-421,2,0,0.726774,14.570497,7.570793
11173,1484,-0.019955,-0.046792,-0.052502,7.567975,321,2623785,-421,2,0,0.754421,5.697810,7.570793
11405,1509,0.041166,-2.336989,-0.007458,4.888116,-321,2668948,421,2,0,0.655462,1.676522,4.890785


In [27]:
# Non ci sono mai piu' di due tracce per indice di collisione e vertice 
check = selected_tracks.groupby(["fIndexCollisions", "fMainMotherOrigIndex"]).size().sort_values(ascending=False)
print(check)

fIndexCollisions  fMainMotherOrigIndex
0                 2147                    2
517               884311                  2
514               972580                  2
                  2198689                 2
                  2306255                 2
                                         ..
244               1712891                 2
                  1755093                 2
                  1912797                 2
245               399366                  2
1509              2668948                 2
Length: 2433, dtype: int64


In [28]:
# Cosa curiosa: due collisioni nello stesso vertice con una D0 e una anti-D0
selected_tracks[selected_tracks["fMainMotherOrigIndex"]==1329085]

,fIndexCollisions,fX,fAlpha,fY,fZ,fPdgCode,fMainMotherOrigIndex,fMainHfMotherPdgCode,fMainMotherNfinalStateDaught,fMainBeautyAncestorPdgCode,fEta,fPt,fPosZ
5740,765,-0.035617,-0.125410,-0.013729,-0.878824,321,1329085,-421,2,0,-0.443050,1.116657,-0.873861
5741,765,0.000954,2.142665,0.036918,-0.883404,-211,1329085,-421,2,0,-0.444506,0.812265,-0.873861
5951,778,0.036049,-2.274048,-0.004173,5.076732,211,1329085,421,2,0,0.761106,2.204355,5.077484
5952,778,0.033269,-2.832733,0.013426,5.078570,-321,1329085,421,2,0,0.530344,4.081808,5.077484


In [35]:
reco_tracks = len(check)
reco_tracks

2433

In [30]:
tot_decays

2655

In [31]:
filtered_tracks_Pt = []

for data_frame in filtered_tracks:
    data_frame["interval"] = pd.cut(data_frame['fPt'], bins=Pt_intervals, labels=Pt_labels, right=True)
    r = data_frame.groupby("interval", observed=True).size()
    filtered_tracks_Pt.append(r)

Pt_int_tot = reduce(lambda x, y: x.add(y, fill_value=0), filtered_tracks_Pt)

In [32]:
print(Pt_int_tot,"\n")
print("tot:", sum(Pt_int_tot))

interval
0-1     776.0
1-2     961.0
2-3     414.0
3-5     276.0
5-8      85.0
8-41     30.0
dtype: float64 

tot: 2542.0


## Generated particles

In [19]:
# file by file...
all_files = file.keys(filter_name = r"DF_*")
file_ID = list(set([code.split("/")[0] for code in all_files]))
# remove the directoryes code
file_ID = [s for s in file_ID if not s.endswith(";1")]


gen_particles = []

gen_D_0 = 0



for directory in file_ID:

    # merge the genparticles an mccollision dataframes on fIndexMcCollisions
    generated_part = file[directory + "/O2genparticles"].arrays(library="pd")
    mc_collision_df = file[directory + "/O2mccollision"].arrays(library="pd")
    generated_part = pd.merge(left=generated_part, right=mc_collision_df, how='inner', left_on='fIndexMcCollisions', right_index=True)
    
    # fPosZ cut
    generated_part = generated_part[generated_part["fPosZ"].abs() < 10]

    # Pseudorapidity cut
    generated_part = generated_part[generated_part["fMainMotherY"].abs() < acceptance]
    
    # Check on the fMainBeautyAncestor = 0
    generated_part = generated_part[generated_part["fMainBeautyAncestorPdgCode"] == 0]

    # Check the identity of the mother particle 
    generated_part = generated_part[generated_part["fPdgCode"].isin([421, -421])]

    # count the generated particles selected after the cuts file by file
    n_gen_D_0 = len(generated_part)
    #print("In this file there are", n_gen_D_0, "D_0 generated")

    # Update the total number of generated particles
    gen_D_0 += n_gen_D_0 

    gen_particles.append(generated_part)
    
    #defined pT interval in which you can/want to “make the measurement”
    # For now no selection on pT...TO DO LATER 
    
    #results
    #good_gen_part = pd.concat([good_gen_part, generated_part], ignore_index=True) 
    #tot_gen_part = pd.concat([tot_gen_part, generated_part], ignore_index=True) 

In [20]:
gen_D_0

21656

In [21]:
gen_particles[0].head(10)

,fPdgCode,fIndexMcCollisions,fMainBeautyAncestorPdgCode,fMainMotherPt,fMainMotherY,fMaxEtaDaughter,fMainBeautyAncestorPt,fMainBeautyAncestorY,fIndexBCs,fGeneratorsID,fPosX,fPosY,fPosZ,fT,fWeight,fImpactParameter
53,-421,15,0,0.519851,-0.395327,0.607354,0.0,0.0,14,12415,-0.033590,-0.019577,0.533055,6.902033e+13,1.0,0.0
100,-421,40,0,1.648214,-0.191849,2.324544,0.0,0.0,39,10367,-0.035183,-0.022749,1.100882,6.902033e+13,1.0,0.0
196,421,66,0,3.309276,-0.378541,0.405473,0.0,0.0,65,2175,-0.028540,-0.023560,4.238533,6.902033e+13,1.0,0.0
233,421,80,0,5.461976,-0.290596,0.519378,0.0,0.0,79,10367,-0.047409,-0.026106,-5.438667,6.902033e+13,1.0,0.0
262,-421,90,0,0.956255,-0.593826,0.881119,0.0,0.0,88,10367,-0.032165,-0.023582,0.492156,6.902033e+13,1.0,0.0
302,-421,100,0,2.357506,-0.723565,0.754387,0.0,0.0,98,10367,-0.026846,-0.026945,4.192909,6.902033e+13,1.0,0.0
415,-421,135,0,1.685175,-0.463881,0.613571,0.0,0.0,133,12415,-0.032282,-0.025073,-2.529209,6.902033e+13,1.0,0.0
446,421,148,0,6.286566,0.276687,0.729227,0.0,0.0,146,2175,-0.027115,-0.022520,6.984261,6.902033e+13,1.0,0.0
633,-421,210,0,3.640128,0.104321,2.121101,0.0,0.0,207,10367,-0.045137,-0.025542,0.323298,6.902033e+13,1.0,0.0
694,421,230,0,5.345599,0.566207,0.651672,0.0,0.0,227,10367,-0.019495,-0.023841,9.882645,6.902033e+13,1.0,0.0


In [22]:
efficiency = tot_decays/gen_D_0
print("Raw efficiency: ", efficiency)

Raw efficiency:  0.058690432212781675


In [1]:
gen_D0_Pt = []

for data_frame in gen_particles:
    data_frame["interval"] = pd.cut(data_frame['fMainMotherPt'], bins=Pt_intervals, labels=Pt_labels, right=True)
    r = data_frame.groupby("interval", observed=True).size()
    gen_D0_Pt.append(r)

tot_D0_Pt = reduce(lambda x, y: x.add(y, fill_value=0), gen_D0_Pt)

NameError: name 'gen_particles' is not defined

In [24]:
print(tot_D0_Pt,"\n")
print("tot:", sum(tot_D0_Pt))
max_Pt_list = [df["fMainMotherPt"].max() for df in gen_particles]
max_Pt = max(max_Pt_list)

# Check we are not loosing particles
if (sum(tot_D0_Pt) < gen_D_0): print("We are loosing paticles...enlarge the last interval up to", max_Pt)

interval
0-1     4558.0
1-2     6528.0
2-3     4441.0
3-5     4044.0
5-8     1525.0
8-41     559.0
dtype: float64 

tot: 21655.0
We are loosing paticles...enlarge the last interval up to 43.902244567871094
